In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =========================
# Cell 1
# Load all extracted data
# =========================

import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/Plan A/data/mimiciv"

# Load files
cohort = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_cohort.csv"))
vitals = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_vitals_24h.csv"))
creat = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_creatinine_24h.csv"))
lact = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_lactate_24h.csv"))
labs_core = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_labs_core_24h.csv"))
labs_kbun = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_labs_k_bun_24h.csv"))
inter = pd.read_csv(os.path.join(BASE_PATH, "mimiciv_interventions_24h.csv"))

print("Loaded all datasets")

Loaded all datasets


In [4]:
# =========================
# Cell 2
# Merge step-by-step
# =========================

df = cohort.copy()

# Merge vitals
df = df.merge(vitals, on=["subject_id","hadm_id","stay_id"], how="left")

# Merge labs
df = df.merge(creat, on=["subject_id","hadm_id","stay_id"], how="left")
df = df.merge(lact, on=["subject_id","hadm_id","stay_id"], how="left")
df = df.merge(labs_core, on=["subject_id","hadm_id","stay_id"], how="left")
df = df.merge(labs_kbun, on=["subject_id","hadm_id","stay_id"], how="left")

# Merge interventions
df = df.merge(inter, on=["subject_id","hadm_id","stay_id"], how="left")

print("Final shape:", df.shape)
df.head()

Final shape: (65366, 71)


,subject_id,hadm_id,stay_id,intime_x,outtime_x,gender_x,age_x,hospital_expire_flag_x,intime_y,outtime_y,...,k_n,k_min,k_max,k_mean,bun_n,bun_min,bun_max,bun_mean,ventilation,vasopressor
0,10002430,26295318,38392119,2129-06-13 00:43:08,2129-06-15 22:51:40,M,90,0,2129-06-13 00:43:08,2129-06-15 22:51:40,...,3,2.8,3.8,3.466667,3,44.0,47.0,45.0,0,0
1,10003502,29011269,35796366,2169-08-26 21:30:32,2169-08-27 22:27:21,F,94,0,2169-08-26 21:30:32,2169-08-27 22:27:21,...,1,4.1,4.1,4.100000,1,56.0,56.0,56.0,0,1
2,10008454,20291550,31959184,2110-11-30 17:11:36,2110-12-05 16:48:24,F,26,0,2110-11-30 17:11:36,2110-12-05 16:48:24,...,1,4.0,4.0,4.000000,1,11.0,11.0,11.0,0,0
3,10009035,28324362,38507547,2161-04-27 10:38:12,2161-04-28 15:06:17,M,28,0,2161-04-27 10:38:12,2161-04-28 15:06:17,...,3,3.8,4.2,3.933333,2,11.0,11.0,11.0,1,1
4,10010867,22429197,39880770,2147-12-30 09:33:00,2148-01-08 18:14:21,F,28,0,2147-12-30 09:33:00,2148-01-08 18:14:21,...,1,4.1,4.1,4.100000,1,8.0,8.0,8.0,0,0


In [5]:
# =========================
# Cell 3
# Sanity checks
# =========================

print("Unique stay_ids:", df["stay_id"].nunique())
print("Total rows:", df.shape[0])

# Should be equal

Unique stay_ids: 65366
Total rows: 65366


In [6]:
# =========================
# Cell 4
# Missingness overview
# =========================

missing = df.isnull().mean().sort_values(ascending=False)
missing.head(20)

,0
lact_min,0.459459
lact_mean,0.459459
lact_max,0.459459
map_max,0.094300
map_min,0.094300
map_mean,0.094300
dbp_max,0.093810
dbp_mean,0.093810
dbp_min,0.093810
sbp_min,0.093780


In [7]:
# =========================
# Cell 5
# Save final dataset
# =========================

save_path = "/content/drive/MyDrive/Plan A/final/mimiciv_final_dataset.csv"
df.to_csv(save_path, index=False)

print("Saved final dataset:", save_path)

Saved final dataset: /content/drive/MyDrive/Plan A/final/mimiciv_final_dataset.csv


In [8]:
# =========================
# Cell 6
# Clean duplicate columns
# =========================

# Keep only one version (from cohort)
df = df.rename(columns={
    "intime_x": "intime",
    "outtime_x": "outtime",
    "gender_x": "gender",
    "age_x": "age",
    "hospital_expire_flag_x": "mortality"
})

# Drop duplicate columns
drop_cols = [col for col in df.columns if col.endswith("_y")]
df = df.drop(columns=drop_cols)

print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (65366, 66)


,subject_id,hadm_id,stay_id,intime,outtime,gender,age,mortality,hr_n,hr_min,...,k_n,k_min,k_max,k_mean,bun_n,bun_min,bun_max,bun_mean,ventilation,vasopressor
0,10002430,26295318,38392119,2129-06-13 00:43:08,2129-06-15 22:51:40,M,90,0,27,66.0,...,3,2.8,3.8,3.466667,3,44.0,47.0,45.0,0,0
1,10003502,29011269,35796366,2169-08-26 21:30:32,2169-08-27 22:27:21,F,94,0,22,35.0,...,1,4.1,4.1,4.100000,1,56.0,56.0,56.0,0,1
2,10008454,20291550,31959184,2110-11-30 17:11:36,2110-12-05 16:48:24,F,26,0,25,92.0,...,1,4.0,4.0,4.000000,1,11.0,11.0,11.0,0,0
3,10009035,28324362,38507547,2161-04-27 10:38:12,2161-04-28 15:06:17,M,28,0,26,90.0,...,3,3.8,4.2,3.933333,2,11.0,11.0,11.0,1,1
4,10010867,22429197,39880770,2147-12-30 09:33:00,2148-01-08 18:14:21,F,28,0,29,87.0,...,1,4.1,4.1,4.100000,1,8.0,8.0,8.0,0,0


In [9]:
# =========================
# Cell 7
# Save cleaned dataset
# =========================

save_path = "/content/drive/MyDrive/Plan A/final/mimiciv_final_dataset_clean.csv"
df.to_csv(save_path, index=False)

print("Saved cleaned dataset:", save_path)

Saved cleaned dataset: /content/drive/MyDrive/Plan A/final/mimiciv_final_dataset_clean.csv
